# 🧠 JORINOVA NEXUS — train ANY lab vision model (one notebook)

Same flow as malaria, but for **any** domain. Set `DOMAIN_KEY`, point at a dataset,
train — it saves to **`backend/models/<DOMAIN_KEY>/<DOMAIN_KEY>.pt`** and the app auto-loads it.

Runtime → **Change runtime type → T4 GPU**. Run cells top to bottom.


## 1. GPU


In [ ]:
!nvidia-smi -L || echo 'set Runtime > T4 GPU and rerun'


## 2. Install


In [ ]:
%pip -q install ultralytics roboflow


## 3. Choose the model key
Must match a registry key (see backend/models/README.md): `parasitology`, `pbs`,
`leukemia`, `anemia`, `trypanosoma`, `leishmania`, `microfilaria`, `gram`, `tb_afb`,
`fungi`, `cytology`, `histology`, `cancer`, `urine` …


In [ ]:
DOMAIN_KEY = 'parasitology'   # ✏️ change to your domain
print('training model:', DOMAIN_KEY)


## 4. Get a labelled dataset — pick ONE
You need a **detection** dataset in YOLO format (images + `data.yaml`).


### Option A — Roboflow (needs your API key)


In [ ]:
# from getpass import getpass; import os
# os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow API key: ').strip()
# WORKSPACE='ws'; PROJECT='proj'; VERSION=1
# from roboflow import Roboflow
# ds = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY']).workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8', location='/content/data')
# DATA_YAML = ds.location + '/data.yaml'


### Option B — Kaggle (needs kaggle.json)
Kaggle → Account → **Create New API Token** downloads `kaggle.json`; upload it when asked.
Many malaria/parasite/blood-cell sets live here (e.g. *Chula-ParasiteEgg-11*, *BCCD*).


In [ ]:
# from google.colab import files; files.upload()   # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle && kaggle datasets download -d <owner>/<dataset> -p /content/data --unzip
# If not already YOLO format, convert to images/ + labels/ + data.yaml (see ml/malaria/prepare_bbbc041.py pattern).
# DATA_YAML = '/content/data/data.yaml'


### Option C — direct URL (public zip already in YOLO format)


In [ ]:
# import urllib.request, zipfile
# urllib.request.urlretrieve('<https URL>.zip', '/content/d.zip')
# zipfile.ZipFile('/content/d.zip').extractall('/content/data')
# DATA_YAML = '/content/data/data.yaml'


## 5. Train
Fill `DATA_YAML` from the option you used above, then run.


In [ ]:
EPOCHS = 80
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics'], check=True); from ultralytics import YOLO
assert 'DATA_YAML' in globals(), 'Set DATA_YAML in step 4 first (uncomment one option).'
model = YOLO('yolov8s.pt')
model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16, name=DOMAIN_KEY,
            patience=20, plots=True, degrees=180, fliplr=0.5, flipud=0.5, hsv_s=0.6)
m = model.val(); print(f'mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}')


## 6. Save as `<DOMAIN_KEY>.pt` and download


In [ ]:
import shutil, glob, os
best = sorted(glob.glob(f'/content/runs/detect/{DOMAIN_KEY}*/weights/best.pt'), key=os.path.getmtime)[-1]
os.makedirs(f'/content/{DOMAIN_KEY}', exist_ok=True)
dest = f'/content/{DOMAIN_KEY}/{DOMAIN_KEY}.pt'
shutil.copy(best, dest); print('saved:', dest)
from google.colab import files; files.download(best)


## 7. Put it in the app
Move the downloaded `best.pt` → **`backend/models/<DOMAIN_KEY>/<DOMAIN_KEY>.pt`**, then
`git add -f backend/models/<DOMAIN_KEY>/<DOMAIN_KEY>.pt && git commit && git push` (merge to main).
The vision service auto-loads it by test type. Ensure `ultralytics` is in the backend requirements.

For **classification** datasets (folders per class, e.g. PBS cells, cancer histology) use
`python ml/malaria/train_classify.py --data <folder>` instead — a classifier head.
